# Дополнительный блок: прогноз нарушения SLA

Необязательная часть итоговой работы. Задача - сравнить базовую логистическую регрессию и случайный лес по метрикам качества.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

candidates=[Path.cwd(),Path.cwd().parent]
ROOT=next((p for p in candidates if (p/'data'/'processed'/'ecommerce_datamart.csv').exists()),Path.cwd())
df=pd.read_csv(ROOT/'data'/'processed'/'ecommerce_datamart.csv')
df=df[(df['sla_eligible']==1)&df['delivery_type'].isin(['standard','express'])].copy()

In [2]:
features=['delivery_type','region_name','category','channel','promised_days','quantity','discount_pct','order_year','order_quarter']
X=df[features]; y=df['late_flag']
cat_cols=['delivery_type','region_name','category','channel','order_quarter']
num_cols=['promised_days','quantity','discount_pct','order_year']
pre=ColumnTransformer([
 ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat_cols),
 ('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols)
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
models={
 'Logistic Regression':LogisticRegression(max_iter=500,class_weight='balanced',random_state=42),
 'Random Forest':RandomForestClassifier(n_estimators=180,max_depth=10,min_samples_leaf=10,class_weight='balanced',random_state=42,n_jobs=-1)
}
for name,model in models.items():
    pipe=Pipeline([('pre',pre),('model',model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    prob=pipe.predict_proba(X_test)[:,1]
    print()
    print(name,'ROC-AUC:',roc_auc_score(y_test,prob))
    print(classification_report(y_test,pred))


Logistic Regression ROC-AUC: 0.666604080616666
              precision    recall  f1-score   support

           0       0.80      0.59      0.68      4921
           1       0.42      0.66      0.51      2183

    accuracy                           0.61      7104
   macro avg       0.61      0.63      0.60      7104
weighted avg       0.68      0.61      0.63      7104




Random Forest ROC-AUC: 0.6632732584826517
              precision    recall  f1-score   support

           0       0.80      0.54      0.65      4921
           1       0.40      0.70      0.51      2183

    accuracy                           0.59      7104
   macro avg       0.60      0.62      0.58      7104
weighted avg       0.68      0.59      0.61      7104



## Интерпретация

Сравнивайте модели не только по accuracy. При несбалансированном целевом признаке важны recall проблемного класса, F1 и ROC-AUC. В отчете обязательно укажите ограничения: синтетические данные, отсутствие части операционных факторов и невозможность трактовать модель как причинную.